In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import scipy as sp
from scipy.linalg import eigh
import umap
import numpy as np
import ot
import hdbscan
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.neighbors import kneighbors_graph
from collections import namedtuple
from Cebra_dim_R import CEBRAAnalysis

In [ ]:
def compute_mmd(K_XX, K_YY, K_XY, gamma):
    """ Computes MMD between two distributions given their kernel matrices. """
    
    K_XX = np.exp(-(K_XX ** 2) * gamma)
    K_YY = np.exp(-(K_YY ** 2) * gamma)
    K_XY = np.exp(-(K_XY ** 2) * gamma)
    m = K_XX.shape[0]  # Number of samples in X
    n = K_YY.shape[0]  # Number of samples in Y
    
    term_X = (1 / (m * (m-1))) * np.sum(K_XX - np.diag(np.diag(K_XX)))
    term_Y = (1 / (n * (n-1))) * np.sum(K_YY - np.diag(np.diag(K_YY)))
    term_XY = (2 / (m * n)) * np.sum(K_XY)

    mmd_squared = term_X + term_Y - term_XY
    #print(mmd_squared)
    return mmd_squared



In [5]:
def geodesic_distance_matrix(points, eps=1e-12):
    """
    Project points onto the unit sphere, then compute the pairwise
    geodesic distance matrix.

    Parameters
    ----------
    points : array-like, shape (n_points, dim)
        Input points.
    eps : float
        Small value to avoid division by zero.

    Returns
    -------
    unit_points : ndarray, shape (n_points, dim)
        Points projected onto the unit sphere.
    D : ndarray, shape (n_points, n_points)
        Pairwise geodesic distance matrix, in radians.
    """
    points = np.asarray(points, dtype=float)

    # Project onto unit sphere
    norms = np.linalg.norm(points, axis=1, keepdims=True)
    unit_points = points / np.clip(norms, eps, None)

    # Cosine of pairwise angles
    cos_theta = unit_points @ unit_points.T

    # Numerical safety
    cos_theta = np.clip(cos_theta, -1.0, 1.0)

    # Geodesic distance on unit sphere
    D = np.arccos(cos_theta)

    return D

In [ ]:


def compute_WassKernel_stratified_improved(data, n_quantile = 100, metric='Euclidean',normalize=False):
    n = len(data)
    # _, quantiles, n_align, n_quantile = compute_WassKernel_stratified(data, n_align = n_quantile, n_quantile = n_quantile, metric='Euclidean',normalize=False, return_distance=True, return_shape=True)
    # for i in range(n):
    #     quantiles[i] = quantiles[i].reshape((n_align, n_quantile))


    quantiles = [0]*n
    for i in range(n):
        if metric == 'Euclidean':
            C1 = sp.spatial.distance.cdist(data[i], data[i])
        elif metric == 'Geodesic'
            C1 = geodesic_distance_matrix(data[i])
        if normalize:
            C1 = C1/np.median(C1)
        n_align = C1.shape[0]
        if n_quantile > 100:
            n_quantile = 100
        # quantiles[i] = distance_matrix_quantiles(C1, n_1=n_align, n_2=n_quantile)
        quantiles[i] = np.zeros((C1.shape[0],n_quantile+1))
        for j in range(C1.shape[0]):
            quantiles[i][j,:] = np.quantile(C1[j,:], np.linspace(0, 1, n_quantile+1))
            
        
    
    distances = np.zeros((n,n))
    for i in range(n):
        for j in range(i+1,n):
            print(i)
            print(j)
            print(quantiles[i].shape)
            print(quantiles[j].shape)
            M = sp.spatial.distance.cdist(quantiles[i], quantiles[j])
            
            a = np.ones((quantiles[i].shape[0],))
            a = a/a.sum()
            b = np.ones((quantiles[j].shape[0],))
            b = b/b.sum()
            distances[i,j] = ot.emd2(a, b, M)
            distances[j,i] = distances[i,j]


    return distances

In [ ]:
def compute_kernel_matrix(data, normalize=False, n_align=100, metric='Euclidean',norm_const=100):
    n = len(data)
    print(n)
    if norm_const is None:
        C1=sp.spatial.distance.cdist(data[0], data[0])
        norm_const = C1.max()
    K = np.zeros( (n,n))
    K = compute_WassKernel_stratified_improved(data, metric=metric,normalize=normalize,n_quantile=n_align)
            
    return K

In [9]:
import plotly.graph_objects as go

def plot_points_3d(points,index, mode="markers", marker_size=4, show_axes=True, title="3D Points"):
    points = np.asarray(points)
    assert points.ndim == 2 and points.shape[1] == 3, "Expected array of shape (N, 3)."

    fig = go.Figure(
        data=[
            go.Scatter3d(
                x=points[:, 0],
                y=points[:, 1],
                z=points[:, 2],
                mode=mode,  # "markers", "lines", or "lines+markers"
                marker=dict(size=marker_size),
            )
        ]
    )

    fig.update_layout(
        title=title,
        margin=dict(l=0, r=0, t=40, b=0),
        scene=dict(
            xaxis=dict(visible=show_axes),
            yaxis=dict(visible=show_axes),
            zaxis=dict(visible=show_axes),
            aspectmode="data",  # keeps proportions faithful to data units
        ),
    )
    fig.show()
    fig.write_html(f'C:\\Users\\zziyu\\Desktop\\CEBRA\\.venv\\Data\\Deven_Data\\plotly_emb_{index}.html')
    return fig

In [50]:

def sample_sphere_surface(N: int, n: int, d: int = 3, rng=None) -> np.ndarray:
    """
    Generate N collections of n iid samples uniformly from the surface
    of the unit sphere S^(d-1) in R^d.

    Returns
    -------
    samples : ndarray, shape (N, n, d)
        samples[i, j] is one point on the sphere.
    """
    rng = np.random.default_rng(rng)

    # Draw standard normal vectors
    x = rng.normal(size=(N, n, d))

    # Normalize each vector to length 1
    norms = np.linalg.norm(x, axis=-1, keepdims=True)
    samples = x / norms

    return samples



def sample_torus_surface(
    N: int,
    n: int,
    R: float = 2.0,
    r: float = 1.0,
    rng=None,
) -> np.ndarray:
    """
    Generate N collections of n iid samples uniformly from the surface
    of a torus in R^3.

    The torus has major radius R and minor radius r, with R > r > 0.

    Returns
    -------
    samples : ndarray, shape (N, n, 3)
        samples[i, j] is one point on the torus surface.
    """
    if not (R > r > 0):
        raise ValueError("Require R > r > 0.")

    rng = np.random.default_rng(rng)

    total = N * n

    # u is uniform around the main circle
    u = rng.uniform(0, 2 * np.pi, size=total)

    # v is NOT uniform for uniform surface area.
    # Its density is proportional to R + r cos(v).
    v_samples = []

    while sum(len(chunk) for chunk in v_samples) < total:
        m = total - sum(len(chunk) for chunk in v_samples)

        v_candidate = rng.uniform(0, 2 * np.pi, size=2 * m)
        accept_prob = (R + r * np.cos(v_candidate)) / (R + r)

        accepted = v_candidate[rng.uniform(size=2 * m) < accept_prob]
        v_samples.append(accepted)

    v = np.concatenate(v_samples)[:total]

    x = (R + r * np.cos(v)) * np.cos(u)
    y = (R + r * np.cos(v)) * np.sin(u)
    z = r * np.sin(v)

    samples = np.stack([x, y, z], axis=-1)
    return samples.reshape(N, n, 3)




def sample_cube_surface(
    N: int,
    n: int,
    side_length: float = 2.0,
    rng=None,
) -> np.ndarray:
    """
    Generate N collections of n iid samples uniformly from the surface
    of a cube centered at the origin.

    Returns
    -------
    samples : ndarray, shape (N, n, 3)
        samples[i, j] is one point on the cube surface.
    """
    if side_length <= 0:
        raise ValueError("side_length must be positive.")

    rng = np.random.default_rng(rng)

    total = N * n
    half = side_length / 2

    # Pick one of the 6 faces uniformly
    faces = rng.integers(0, 6, size=total)

    # Pick two coordinates uniformly on the chosen face
    coords = rng.uniform(-half, half, size=(total, 3))

    # Set one coordinate to +/- half depending on the face
    axis = faces // 2          # 0 for x, 1 for y, 2 for z
    sign = 2 * (faces % 2) - 1 # -1 or +1

    coords[np.arange(total), axis] = sign * half

    return coords.reshape(N, n, 3)


def sample_triangle_uniform(a, b, c, m: int, rng) -> np.ndarray:
    """
    Uniformly sample m points from the triangle with vertices a, b, c.
    """
    u = rng.uniform(size=m)
    v = rng.uniform(size=m)

    # Reflect points outside the unit simplex
    mask = u + v > 1
    u[mask] = 1 - u[mask]
    v[mask] = 1 - v[mask]

    return a + u[:, None] * (b - a) + v[:, None] * (c - a)


def sample_tetrahedron_surface(
    N: int,
    n: int,
    side_length: float = 2.0,
    rng=None,
) -> np.ndarray:
    """
    Generate N collections of n iid samples uniformly from the surface
    of a regular triangular pyramid, i.e. a regular tetrahedron.

    Returns
    -------
    samples : ndarray, shape (N, n, 3)
    """
    if side_length <= 0:
        raise ValueError("side_length must be positive.")

    rng = np.random.default_rng(rng)
    total = N * n

    # Vertices of a regular tetrahedron centered at the origin.
    vertices = np.array([
        [1,  1,  1],
        [1, -1, -1],
        [-1, 1, -1],
        [-1, -1, 1],
    ], dtype=float)

    # Scale to desired side length.
    # The original side length is 2 * sqrt(2).
    vertices *= side_length / (2 * np.sqrt(2))

    faces = np.array([
        [0, 1, 2],
        [0, 1, 3],
        [0, 2, 3],
        [1, 2, 3],
    ])

    # For a regular tetrahedron, all 4 faces have equal area.
    chosen_faces = rng.integers(0, 4, size=total)

    samples = np.empty((total, 3))

    for f in range(4):
        idx = np.where(chosen_faces == f)[0]
        if len(idx) == 0:
            continue

        a, b, c = vertices[faces[f]]
        samples[idx] = sample_triangle_uniform(a, b, c, len(idx), rng)

    return samples.reshape(N, n, 3)

def sample_three_surfaces(
    N: int,
    n: int,
    torus_R: float = 2.0,
    torus_r: float = 1.0,
    cube_side_length: float = 2.0,
    rng=None,
) -> np.ndarray:
    """
    Generate N collections of n samples from each of:
    sphere surface, torus surface, cube surface.

    Returns
    -------
    samples : ndarray, shape (3 * N, n, 3)

        samples[0:N]       are sphere samples
        samples[N:2*N]     are torus samples
        samples[2*N:3*N]   are cube samples
    """
    rng = np.random.default_rng(rng)

    sphere = sample_sphere_surface(N, n, rng=rng)
    torus = sample_torus_surface(N, n, R=torus_R, r=torus_r, rng=rng)
    cube = sample_cube_surface(N, n, side_length=cube_side_length, rng=rng)

    return np.concatenate([sphere, torus, cube], axis=0)

from scipy.spatial.transform import Rotation

def apply_random_rotations(samples: np.ndarray, rng=None) -> np.ndarray:
    """
    Apply a different random SO(3) rotation to each sample group.

    Parameters
    ----------
    samples : ndarray, shape (num_groups, n, 3)

    Returns
    -------
    rotated : ndarray, shape (num_groups, n, 3)
    """
    if samples.ndim != 3 or samples.shape[-1] != 3:
        raise ValueError("samples must have shape (num_groups, n, 3).")

    rng = np.random.default_rng(rng)

    num_groups = samples.shape[0]

    rotations = Rotation.random(num_groups, random_state=rng)
    R = rotations.as_matrix()  # shape (num_groups, 3, 3)

    # For each group g and point i:
    # rotated[g, i] = R[g] @ samples[g, i]
    rotated = np.einsum("gij,gnj->gni", R, samples)

    return rotated



def sample_four_surfaces(
    N: int,
    n: int,
    torus_R: float = 2.0,
    torus_r: float = 1.0,
    cube_side_length: float = 2.0,
    tetra_side_length: float = 2.0,
    random_rotate: bool = True,
    rng=None,
) -> np.ndarray:
    """
    Generate N collections of n samples from each of 4 surfaces:

    1. sphere
    2. torus
    3. cube
    4. triangular pyramid / tetrahedron

    Optionally applies a different random SO(3) rotation to every group.

    Returns
    -------
    samples : ndarray, shape (4 * N, n, 3)
    """
    rng = np.random.default_rng(rng)

    sphere = sample_sphere_surface(N, n, rng=rng)

    torus = sample_torus_surface(
        N,
        n,
        R=torus_R,
        r=torus_r,
        rng=rng,
    )

    cube = sample_cube_surface(
        N,
        n,
        side_length=cube_side_length,
        rng=rng,
    )

    tetra = sample_tetrahedron_surface(
        N,
        n,
        side_length=tetra_side_length,
        rng=rng,
    )

    samples = np.concatenate([sphere, torus, cube, tetra], axis=0)

    if random_rotate:
        samples = apply_random_rotations(samples, rng=rng)

    return samples



In [ ]:
def extract_embeddings(data_path, embedding_folder_path):
    cebrarun = CEBRAAnalysis(session_choose = True, data_path = data_path)
    cebrarun.run_analysis(embedding_folder_path = embedding_folder_path)
return None


In [ ]:
def cluster(D, min_cluster_size, min_samples):

    clusterer = hdbscan.HDBSCAN(
        metric="precomputed",
        min_cluster_size=min_cluster_size,  
        min_samples=min_samples,         
        cluster_selection_method='leaf'
    )

    clusters = clusterer.fit_predict(D)
    cluster_labels = np.unique(clusters)    
    cluster_indices = {
    c: np.where(clusters == c)[0]
    for c in cluster_labels
    }
    return cluster_labels, cluster_indices

[ 2  2  2 -1  2  2  2  2  2  2  2  2  2  2 -1  2  2  2  2  2  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  3 -1 -1  3  3  3 -1 -1
  3  3  3  3  3  3  3  3  3  3  3  3  1  1  1  1  1  1  1  1  1  1  1  1
  1  1  1  1  1  1  1  1]


In [ ]:
def median_heuristic(X):
    """
    Estimate RBF bandwidth using the median pairwise distance heuristic.
    """
    X_norm = np.sum(X ** 2, axis=1)[:, None]
    sq_dists = X_norm + X_norm.T - 2 * X @ X.T

    # Remove diagonal zeros
    sq_dists = sq_dists[np.triu_indices_from(sq_dists, k=1)]

    median_sq_dist = np.median(sq_dists)

    if median_sq_dist <= 0:
        median_sq_dist = 1.0

    gamma = 1.0 / (2.0 * median_sq_dist)
    return gamma





def mmd_permutation_test(X, Y, n_permutations=1000, gamma = None, random_state=None):
    """
    Perform a two-sample MMD permutation test between X and Y.
    """
    rng = np.random.default_rng(random_state)

    X = np.asarray(X)
    Y = np.asarray(Y)

    n_x = len(X)
    n_y = len(Y)

    Z = np.vstack([X, Y])
    if gamma is None:
        gamma = median_heuristic(Z)


    observed_mmd = compute_mmd(X, Y, gamma)

    permuted_mmds = np.empty(n_permutations)

    for i in range(n_permutations):
        permuted_indices = rng.permutation(n_x + n_y)

        X_perm = Z[permuted_indices[:n_x]]
        Y_perm = Z[permuted_indices[n_x:]]

        permuted_mmds[i] = compute_mmd(X_perm, Y_perm, gamma)

    p_value = (np.sum(permuted_mmds >= observed_mmd) + 1) / (n_permutations + 1)

    return observed_mmd, p_value





def pairwise_cluster_mmd_tests(data, cluster_labels, n_permutations=1000, random_state=None):
    """
    Perform MMD permutation tests between all pairs of clusters.

    Parameters
    ----------
    data : array-like, shape (n_samples, n_features)
        Data points.

    cluster_labels : array-like, shape (n_samples,)
        Cluster assignment for each data point.

    n_permutations : int
        Number of permutations for each test.

    random_state : int or None
        Random seed.

    Returns
    -------
    results : dict
        Dictionary mapping (cluster_a, cluster_b) to MMD statistic and p-value.
    """
    data = np.asarray(data)
    cluster_labels = np.asarray(cluster_labels)

    clusters = np.unique(cluster_labels)

    results = {}

    rng = np.random.default_rng(random_state)

    for c1, c2 in combinations(clusters, 2):
        idx1 = np.where(cluster_labels == c1)[0]
        idx2 = np.where(cluster_labels == c2)[0]

        X = data[idx1]
        Y = data[idx2]

        seed = rng.integers(0, 1_000_000_000)

        mmd_stat, p_value = mmd_permutation_test(
            X,
            Y,
            n_permutations=n_permutations,
            random_state=seed
        )

        results[(c1, c2)] = {
            "mmd2": mmd_stat,
            "p_value": p_value,
            "n_cluster_1": len(X),
            "n_cluster_2": len(Y),
        }

    return results

In [ ]:


def plot_umap_from_cluster_indices(
    data,
    cluster_labels,
    cluster_indices,
    n_neighbors=15,
    min_dist=0.1,
    metric="euclidean",
    random_state=42,
    figsize=(8, 6),
    s=20,
    alpha=0.8,
    title="UMAP plot by cluster",
    plot_noise=True,
):
    """
    Fit UMAP on data and plot clusters using precomputed cluster_indices.

    Parameters
    ----------
    data : array-like, shape (n_samples, n_features)
        Original data matrix.

    cluster_labels : array-like
        Unique cluster labels returned by your cluster() function.

    cluster_indices : dict
        Dictionary mapping cluster label -> sample indices.

    plot_noise : bool
        Whether to plot HDBSCAN noise points labeled as -1.

    Returns
    -------
    embedding : ndarray, shape (n_samples, 2)
        The UMAP embedding.
    """

    data = np.asarray(data)

    reducer = umap.UMAP(
        n_components=2,
        n_neighbors=n_neighbors,
        min_dist=min_dist,
        metric=metric,
        random_state=random_state,
    )

    embedding = reducer.fit_transform(data)

    plt.figure(figsize=figsize)

    for c in cluster_labels:
        if c == -1 and not plot_noise:
            print('Unassigned data points, possible noise')
            continue

        idx = cluster_indices[c]

        label = "Noise" if c == -1 else f"Cluster {c}"

        plt.scatter(
            embedding[idx, 0],
            embedding[idx, 1],
            s=s,
            alpha=alpha,
            label=label,
        )

    plt.xlabel("UMAP 1")
    plt.ylabel("UMAP 2")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()

    return embedding